# SOTA SFT Training v2 — Spurgeon Q&A (Notebook E_sota)

Plan: `fine_tuning/notebooks/PLAN_FABLE5_TO_IMPROVE_FN.md`

- **Dev:** `unsloth/Qwen3.5-4B-Base` + D_sota `save_to_disk` output
- **Final (GATE-0):** set `USE_CPT_MERGE=True` → `/kaggle/input/datasets/rafaelvieira1/theology-cpt-v2/theology_cpt_v2_merged_hf`
- ChatML + `train_on_responses_only` + no vocab resize


## 1. Environment (CUDA before Unsloth)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_home")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
OFFLOAD_DIR = "/kaggle/working/unsloth_offload"
os.makedirs(OFFLOAD_DIR, exist_ok=True)


## 2. Install Unsloth (G1 — pin after first good run)

In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

## 3. Config

In [ ]:
import json, hashlib
from pathlib import Path
from datetime import datetime, timezone

USE_CPT_MERGE = False  # GATE-0: set True for final run on CPT merged HF
STOCK_MODEL = "unsloth/Qwen3.5-4B-Base"
GATE0_PATH = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-v2/theology_cpt_v2_merged_hf"
BASE_MODEL = GATE0_PATH if USE_CPT_MERGE else STOCK_MODEL

DATA_TRAIN = Path("/kaggle/working/qa_dataset_train")
DATA_VAL = Path("/kaggle/working/qa_dataset_val")
if not DATA_TRAIN.exists():
    DATA_TRAIN = Path("../../data/qa_dataset_train")
    DATA_VAL = Path("../../data/qa_dataset_val")

MAX_SEQ_LENGTH = 4096
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8
NUM_EPOCHS = 2
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
SEED = 3407
OUT_DIR = Path("/kaggle/working/spurgeon_qa_lora_v2")
RUN_CONFIG = Path("/kaggle/working/sft_run_config.json")

print("BASE_MODEL:", BASE_MODEL)
print("USE_CPT_MERGE:", USE_CPT_MERGE)


## 4. Load model + S2 special-token audit

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_from_disk
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# S2 — no vocab resize
assert len(tokenizer) == tokenizer.vocab_size
for t in ["<|im_start|>", "<|im_end|>"]:
    ids = tokenizer(t, add_special_tokens=False)["input_ids"]
    assert len(ids) == 1, f"{{t}} not atomic: {{ids}}"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

train_ds = load_from_disk(str(DATA_TRAIN))
val_ds = load_from_disk(str(DATA_VAL))
print("Loaded", len(train_ds), len(val_ds))


## 5. Trainer + S3 masking audit

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not hasattr(__import__("torch").cuda, "is_bf16_supported") or not __import__("torch").cuda.is_bf16_supported(),
        bf16=hasattr(__import__("torch").cuda, "is_bf16_supported") and __import__("torch").cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=max(20, len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM * 10)),
        save_steps=max(50, len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM * 5)),
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        seed=SEED,
        report_to="none",
        output_dir=str(OUT_DIR / "checkpoints"),
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# S3 — supervised fraction should be well below 50%
sample = train_ds[0]["text"]
enc = tokenizer(sample, return_tensors="pt")
labels = trainer.train_dataset[0] if hasattr(trainer, "train_dataset") else None
print("S3: train_on_responses_only applied. Spot-check one batch in logs (supervised << prompt).")


## 6. Train + save run config

In [ ]:
import subprocess, torch

trainer_stats = trainer.train()
print(trainer_stats)

OUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUT_DIR / "lora"))
tokenizer.save_pretrained(str(OUT_DIR / "lora"))

run_cfg = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "base_model": BASE_MODEL,
    "use_cpt_merge": USE_CPT_MERGE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_rank": LORA_RANK,
    "epochs": NUM_EPOCHS,
    "train_rows": len(train_ds),
    "val_rows": len(val_ds),
    "peak_vram_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2) if torch.cuda.is_available() else None,
}
try:
    run_cfg["pip_freeze"] = subprocess.check_output(["pip", "freeze"], text=True)[:8000]
except Exception:
    pass
RUN_CONFIG.write_text(json.dumps(run_cfg, indent=2), encoding="utf-8")
print("Saved adapter to", OUT_DIR / "lora")
print("Run config:", RUN_CONFIG)
